# Olist 数据清洗（PySpark + Parquet）

基于质量检测结果，用 PySpark DataFrame API 对 9 张表做清洗，结果写回 HDFS 成 **Parquet** 格式（列式存储，后续分析读起来更快）。

**清洗方向（对应质量检测发现的问题）**：
1. **去重**：geolocation 26 万完全重复行
2. **类型转换**：STRING → TIMESTAMP / DOUBLE / INT
3. **缺失处理**：products 缺类目补 `unknown`
4. **列名修正**：`product_name_lenght` → `product_name_length`（官方拼写错误）
5. **派生字段**：订单表加「准时与否 / 延误天数」

**输出**：HDFS `/olist/clean/` 目录下的 9 个 Parquet 文件。

## 0. 环境配置

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Olist-DataClean")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

HDFS_DATA = "hdfs://node100:9000/olist/data"     # 原始数据
HDFS_CLEAN = "hdfs://node100:9000/olist/clean"    # 清洗后输出

def read_csv(filename, fields):
    """读原始 CSV（header=true，全 STRING 读入）"""
    schema = StructType([StructField(f, StringType(), True) for f in fields])
    return (
        spark.read.option("header", "true").schema(schema)
        .csv(f"{HDFS_DATA}/{filename}")
    )

print("[OK] SparkSession 就绪，Spark", spark.version)


## 1. 清洗 customers（客户表）
质量检测：无缺失、无重复。仅做类型整理（邮编前缀保持字符串，因为可能有前导 0）。

In [ ]:
df_customers = read_csv("olist_customers_dataset.csv",
    ["customer_id","customer_unique_id","customer_zip_code_prefix","customer_city","customer_state"])

clean_customers = df_customers \
    .dropDuplicates() \
    .filter(F.col("customer_id").isNotNull())

print("清洗前:", df_customers.count(), "清洗后:", clean_customers.count())
clean_customers.write.mode("overwrite").parquet(f"{HDFS_CLEAN}/customers")
print("[OK] customers 已写入 Parquet")


## 2. 清洗 geolocation（重点：去重 + 经纬度转 DOUBLE）
质量检测发现 26 万完全重复行，去重后 100 万 → 73.8 万。

In [ ]:
df_geo = read_csv("olist_geolocation_dataset.csv",
    ["geolocation_zip_code_prefix","geolocation_lat","geolocation_lng","geolocation_city","geolocation_state"])

clean_geo = df_geo \
    .dropDuplicates() \
    .withColumn("geolocation_lat", F.col("geolocation_lat").cast("double")) \
    .withColumn("geolocation_lng", F.col("geolocation_lng").cast("double"))

before = df_geo.count()
after = clean_geo.count()
print(f"清洗前: {before:,}  清洗后: {after:,}  去除重复: {before-after:,}")
clean_geo.write.mode("overwrite").parquet(f"{HDFS_CLEAN}/geolocation")
print("[OK] geolocation 已去重并写入 Parquet")


In [ ]:
# 验证：去重后是否还有重复（应为 0）
dup_check = clean_geo.count() - clean_geo.distinct().count()
print("去重后剩余重复行:", dup_check)


## 3. 清洗 order_items（订单商品，数值转 DOUBLE/INT）

In [ ]:
df_items = read_csv("olist_order_items_dataset.csv",
    ["order_id","order_item_id","product_id","seller_id","shipping_limit_date","price","freight_value"])

clean_items = df_items \
    .dropDuplicates() \
    .withColumn("order_item_id", F.col("order_item_id").cast("int")) \
    .withColumn("price", F.col("price").cast("double")) \
    .withColumn("freight_value", F.col("freight_value").cast("double")) \
    .withColumn("shipping_limit_date", F.col("shipping_limit_date").cast("timestamp"))

print("清洗前:", df_items.count(), "清洗后:", clean_items.count())
clean_items.write.mode("overwrite").parquet(f"{HDFS_CLEAN}/order_items")
print("[OK] order_items 已写入 Parquet")


## 4. 清洗 order_payments（支付，金额转 DOUBLE，not_defined 归入 other）

In [ ]:
df_pay = read_csv("olist_order_payments_dataset.csv",
    ["order_id","payment_sequential","payment_type","payment_installments","payment_value"])

clean_pay = df_pay \
    .dropDuplicates() \
    .withColumn("payment_sequential", F.col("payment_sequential").cast("int")) \
    .withColumn("payment_installments", F.col("payment_installments").cast("int")) \
    .withColumn("payment_value", F.col("payment_value").cast("double")) \
    .withColumn("payment_type",
        F.when(F.col("payment_type") == "not_defined", "other")
         .otherwise(F.col("payment_type")))

print("清洗前:", df_pay.count(), "清洗后:", clean_pay.count())
clean_pay.write.mode("overwrite").parquet(f"{HDFS_CLEAN}/order_payments")
print("[OK] order_payments 已写入 Parquet")


## 5. 清洗 order_reviews（评论，评分转 INT，时间转 TIMESTAMP）
评论文本缺失是正常选填，**保留原样**，不做填充。

In [ ]:
df_rev = read_csv("olist_order_reviews_dataset.csv",
    ["review_id","order_id","review_score","review_comment_title","review_comment_message","review_creation_date","review_answer_timestamp"])

clean_rev = df_rev \
    .dropDuplicates() \
    .withColumn("review_score", F.col("review_score").cast("int")) \
    .withColumn("review_creation_date", F.col("review_creation_date").cast("timestamp")) \
    .withColumn("review_answer_timestamp", F.col("review_answer_timestamp").cast("timestamp"))

print("清洗前:", df_rev.count(), "清洗后:", clean_rev.count())
clean_rev.write.mode("overwrite").parquet(f"{HDFS_CLEAN}/order_reviews")
print("[OK] order_reviews 已写入 Parquet")


## 6. 清洗 orders（订单主表：时间转 TIMESTAMP + 派生准时/延误字段）
这是最关键的表，物流时效分析依赖它。未送达订单的日期缺失是业务正常，**保留**。

In [ ]:
df_orders = read_csv("olist_orders_dataset.csv",
    ["order_id","customer_id","order_status","order_purchase_timestamp","order_approved_at","order_delivered_carrier_date","order_delivered_customer_date","order_estimated_delivery_date"])

clean_orders = df_orders \
    .dropDuplicates() \
    .withColumn("order_purchase_timestamp", F.col("order_purchase_timestamp").cast("timestamp")) \
    .withColumn("order_approved_at", F.col("order_approved_at").cast("timestamp")) \
    .withColumn("order_delivered_carrier_date", F.col("order_delivered_carrier_date").cast("timestamp")) \
    .withColumn("order_delivered_customer_date", F.col("order_delivered_customer_date").cast("timestamp")) \
    .withColumn("order_estimated_delivery_date", F.col("order_estimated_delivery_date").cast("timestamp"))

# 派生字段：延误天数（正=延误，负=提前）+ 是否准时（仅 delivered 有意义）
clean_orders = clean_orders \
    .withColumn("delay_days",
        F.datediff(F.col("order_delivered_customer_date"), F.col("order_estimated_delivery_date"))) \
    .withColumn("is_on_time",
        F.when(F.col("order_delivered_customer_date") <= F.col("order_estimated_delivery_date"), 1)
         .otherwise(0))

print("清洗前:", df_orders.count(), "清洗后:", clean_orders.count())
clean_orders.write.mode("overwrite").parquet(f"{HDFS_CLEAN}/orders")
print("[OK] orders 已写入 Parquet（含派生字段 delay_days / is_on_time）")


## 7. 清洗 products（商品：类目缺失补 unknown + 列名修正 + 数值转换）

In [ ]:
df_prod = read_csv("olist_products_dataset.csv",
    ["product_id","product_category_name","product_name_lenght","product_description_lenght","product_photos_qty","product_weight_g","product_length_cm","product_height_cm","product_width_cm"])

clean_prod = (
    df_prod
    .dropDuplicates()
    .withColumnRenamed("product_name_lenght", "product_name_length")       # 修正官方拼写错误
    .withColumnRenamed("product_description_lenght", "product_description_length")  # 修正官方拼写错误
    .withColumn("product_category_name",
        F.when(F.col("product_category_name").isNull(), "unknown")
         .when(F.col("product_category_name") == "", "unknown")
         .otherwise(F.col("product_category_name")))                         # 缺失类目补 unknown
    .withColumn("product_name_length", F.col("product_name_length").cast("int"))
    .withColumn("product_description_length", F.col("product_description_length").cast("int"))
    .withColumn("product_photos_qty", F.col("product_photos_qty").cast("int"))
    .withColumn("product_weight_g", F.col("product_weight_g").cast("double"))
    .withColumn("product_length_cm", F.col("product_length_cm").cast("double"))
    .withColumn("product_height_cm", F.col("product_height_cm").cast("double"))
    .withColumn("product_width_cm", F.col("product_width_cm").cast("double"))
)

print("清洗前:", df_prod.count(), "清洗后:", clean_prod.count())
print("补 unknown 的类目数:", clean_prod.filter(F.col("product_category_name")=="unknown").count())
clean_prod.write.mode("overwrite").parquet(f"{HDFS_CLEAN}/products")
print("[OK] products 已写入 Parquet")

## 8. 清洗 sellers（卖家）和 category_translation（类目翻译）

In [ ]:
df_seller = read_csv("olist_sellers_dataset.csv",
    ["seller_id","seller_zip_code_prefix","seller_city","seller_state"])
clean_seller = df_seller.dropDuplicates().filter(F.col("seller_id").isNotNull())
clean_seller.write.mode("overwrite").parquet(f"{HDFS_CLEAN}/sellers")
print("sellers:", clean_seller.count(), "行")

df_cat = read_csv("product_category_name_translation.csv",
    ["product_category_name","product_category_name_english"])
clean_cat = df_cat.dropDuplicates()
clean_cat.write.mode("overwrite").parquet(f"{HDFS_CLEAN}/category_translation")
print("category_translation:", clean_cat.count(), "行")


## 9. 清洗结果验证（汇总）
列出清洗后各表的行数，确认全部就绪。

In [ ]:
print("=" * 60)
print("清洗结果汇总（HDFS /olist/clean/）")
print("=" * 60)

clean_tables = [
    ("customers", "customers"),
    ("geolocation", "geolocation"),
    ("order_items", "order_items"),
    ("order_payments", "order_payments"),
    ("order_reviews", "order_reviews"),
    ("orders", "orders"),
    ("products", "products"),
    ("sellers", "sellers"),
    ("category_translation", "category_translation"),
]

for name, _ in clean_tables:
    cnt = spark.read.parquet(f"{HDFS_CLEAN}/{name}").count()
    print(f"  {name:22s} {cnt:>10,} 行")

print("\n[完成] 9 张表已全部清洗并写入 Parquet")
print("下一步：合表（join 成订单宽表）")

spark.stop()
